# November 2, 2026

In [2]:
import scipy.sparse
import numpy as np

In [13]:
def SBM(clusterSize, probabilities, self_loops = True):
    '''
    Sample a random network based on the stochasic block model and return its adjacency matrix in scipy.sparse array format (csr)
    There are clusterNum communities of size clusterSize, with connection probabilities specified by the input parameter probabilities

    clusterSize (int): the size of each cluster, i.e how many nodes are in the cluster
    probabilities (numpy array): dimensions clusterNum by clusterNum
        - probabilities[i,j] is the probability of an egde between a unit in cluster i and a unit in cluster j
        - assumed to be symmetric, i.e. probabilities[i,j] = probabilities[j,i]
    '''
    p = np.kron(probabilities, np.ones((clusterSize,clusterSize))) # create an n by n probability matrix
    n = p.shape[0]
    A = np.random.rand(n,n)
    A = (A < p) + 0
    if self_loops:
        A[range(n),range(n)] = 1   # set diagonal = 1: everyone is connected to itself and affected by their own treatment
    return scipy.sparse.csr_array(A)

In [ ]:
# Written by Claude Sonnet 5.5
def sbm_prob_matrix(p_in, p_out, p_cross, clusterNum, clusterNumBig):
    """
    Parameters
    ----------
    p_in : float
        Edge probability within the same fine (small) cluster.
    p_out : float
        Edge probability between different fine clusters in the same coarse (big) cluster.
    p_cross : float
        Edge probability between fine clusters in different coarse (big) clusters.
    clusterNum : int
        Total number of fine (small) clusters. Must be divisible by clusterNumBig.
    clusterNumBig : int
        Number of coarse (big) clusters.
    """
    if clusterNum % clusterNumBig != 0:
        raise ValueError("clusterNum must be divisible by clusterNumBig")
    m = clusterNum // clusterNumBig                    # fine clusters per coarse cluster

    inner = np.full((m, m), p_out)                     # within a coarse cluster
    np.fill_diagonal(inner, p_in)                      # same fine cluster

    B = np.eye(clusterNumBig)
    cross = np.full((m, m), p_cross)                   # between coarse clusters
    return np.kron(B, inner) + np.kron(1 - B, cross)   # clusterNum x clusterNum

In [27]:
p_out = 0.3
p_in = 0.6
p_cross = 0

clusterNum = 9 # number of small clusters
clusterNumBig = 3 # number of big clusters

m = clusterNum // clusterNumBig

inner = np.full((m, m), p_out)  
np.fill_diagonal(inner, p_in) 

B = np.eye(clusterNumBig)
cross = np.full((m, m), p_cross) 

print(np.kron(B, inner) + np.kron(1 - B, cross) )  

[[0.6 0.3 0.3 0.  0.  0.  0.  0.  0. ]
 [0.3 0.6 0.3 0.  0.  0.  0.  0.  0. ]
 [0.3 0.3 0.6 0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.6 0.3 0.3 0.  0.  0. ]
 [0.  0.  0.  0.3 0.6 0.3 0.  0.  0. ]
 [0.  0.  0.  0.3 0.3 0.6 0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.6 0.3 0.3]
 [0.  0.  0.  0.  0.  0.  0.3 0.6 0.3]
 [0.  0.  0.  0.  0.  0.  0.3 0.3 0.6]]
